# 5장 1강. LLM 워크플로우의 구성과 활용
## 실습문제


## 실습 목표

- 단일 호출, 순차 처리, 조건부 분기의 특징과 선택 이유를 설명할 수 있다.
- 앞 단계의 분류 결과와 원문을 다음 분석의 입력으로 전달할 수 있다.
- 미리 정한 조건에 따라 요약 또는 질의응답 작업을 선택할 수 있다.
- 허용되지 않은 중간 결과가 나오면 후속 작업을 실행하지 않고 원인을 기록할 수 있다.
- 분류 단계 실패와 후속 분석 실패를 구분하고, 원문·중간 결과·최종 결과를 함께 관리할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- CSV 3개를 노트북과 같은 폴더에 둡니다. 다른 위치라면 `DATA_DIR`를 수정합니다.
- 앞 강의의 노트북은 필요하지 않습니다. 이 파일을 위에서 아래로 실행합니다.

| 데이터 | 역할 |
|---|---|
| 엘지유플러스_분류.csv | 상담 요건 분류 지시·원문·기존 출력 확인 |
| 엘지유플러스_요약.csv | 같은 상담의 요약 지시와 기존 출력 확인 |
| 엘지유플러스_질의응답.csv | 같은 상담의 질문과 기존 답변 확인, 질의응답 경로의 입력 준비 |

공통 원본 컬럼은 `source_id`, `consulting_category`, `task_category`, `instruction`, `input`, `output`입니다. `source_id`는 상담 ID, `instruction`은 작업 지시·질문, `input`은 상담 원문, `output`은 CSV에 이미 저장된 출력입니다. 하나의 상담에 여러 작업이 있어 source_id가 반복됩니다.

### 실습 범위와 자료 구분

교안의 명언·리뷰 사례를 **상담 데이터**에 적용합니다. 기초~중급 수준의 필수 2문제와 같은 수준의 과제 1문제로 구성했습니다. 기본 문법과 제공 함수를 이용해 처리 흐름을 확인합니다.

이번 노트북은 **실제 LLM API를 호출하지 않습니다.** 모델이 응답했다고 가정한 값을 제공하고, 그 값을 순서대로 전달하거나 조건에 따라 다음 작업을 선택하는 프로그램을 작성합니다. API 키·유료 계정 없이 실행할 수 있습니다.

- CSV의 원문·지시·기존 출력은 실제 데이터입니다. 기존 output을 자동으로 새 모델 응답으로 사용하지 않습니다.
- 가상 분류·요약·답변·요청 상태는 작성자가 만든 학습 자료입니다. 실제 모델의 성능이나 오류 발생 빈도를 측정한 결과가 아닙니다.
- '가상 후속 단계 실행'은 준비된 응답을 선택해 흐름을 확인한다는 뜻입니다. 파이썬이 상담을 이해해 새 답변을 생성하는 것은 아닙니다.
- 요청 설계용 딕셔너리는 단계별 전달 정보를 표현합니다. 특정 API의 실제 요청 규격은 아닙니다.
- 호출 횟수 비교는 해당 설계를 실제 API로 구현할 때의 **정상 경로 기준 예상 횟수**입니다. 이 노트북의 실제 API 호출 횟수는 0회입니다.
- RAG 검색, Tool Calling, Agent의 자율적 행동 선택, 상세 API 연결, 자동 재시도, 정확도 지표 계산은 포함하지 않습니다.
- 원문과 가림 표시(`<NAME>`, `<DATE>` 등)를 유지합니다. 기존 출력도 원문에 비추어 확인할 참고 자료입니다.

| 교안 내용 | 대응 실습 |
|---|---|
| 단일 호출·순차 처리·조건부 분기 | 필수 1의 방식 비교, 필수 2의 경로 선택 |
| 분석 목적에 따른 워크플로우 선택 | 각 문제의 작업 의존성·필요 결과·시간과 비용 해석 |
| 단계별 입력·출력 연결과 처리 결과 확인 | 원문·분류 결과 전달, 과제 1의 실패 단계 구분 |


## 실습 준비

아래 코드는 제공 코드입니다. 실행 후 결과를 확인하세요. 함수 구현은 평가하지 않습니다.

1. 데이터 불러오기: 세 CSV를 별도 DataFrame으로 읽습니다.
2. 분석 대상 컬럼 확인: 원문·지시·기존 출력과 ID를 확인합니다.
3. 결측치 및 자료형 확인: 결측치·빈 문자열, 원본 컬럼의 자료형을 확인합니다.
4. 기초 통계량과 분포 확인: 고유 상담 수, 작업 유형별 행 수, 본문 길이를 확인합니다.

`record_id`는 파일 이름과 0부터 시작하는 행 위치를 결합한 실습용 행 식별자입니다. 원본의 영구 ID는 아니며 파일 행 순서가 바뀌면 달라집니다. 본문 글자 수는 토큰 수와 다릅니다.


In [28]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])



[분류] 원본 16,080행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
상담 요건    3216
상담 주제    3216
상담 사유    3216
상담 내용    3216
상담 결과    3216
행별 본문 글자 수(토큰 수가 아님):
min      670.00
mean    1583.42
max     2836.00

[요약] 원본 11,690행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
기본 요약          3217
길이 제한 요약       2928
주제 중점 요약       2862
핵심 정보 추출 요약    2683
행별 본문 글자 수(토큰 수가 아님):
min      215.00
mean    1576.97
max     2836.00

[질의응답] 원본 7,497행 × 6열
                       자료형  결측치
source_id        

### 워크플로우 입력 준비 — 제공 코드

상담 40001·40003·40005·40006·40007에서 **상담 요건 분류 한 행, 길이 제한 요약 한 행, 지정 질문 한 행**을 선택합니다. 세 행의 원문이 같은지 확인한 뒤 상담별 한 행의 `documents`로 정리합니다.

- `기존_category`, `기존_summary`, `기존_answer`는 CSV 참고 출력입니다.
- `question`은 이번 실습에서 각 상담에 미리 지정한 질문입니다. 모델이 자율적으로 질문을 고르는 구조가 아닙니다.
- 이번에는 상담마다 워크플로우를 한 번 적용하므로 source_id를 document_id로 사용합니다. 한 상담에 여러 질문을 각각 처리한다면 별도 작업 ID가 필요합니다.
- `get_document()`는 문서 한 행을 선택합니다.
- `make_follow_input()`은 원문·분류 결과·작업 지시를 후속 입력으로 묶기만 합니다. API 요청이나 응답 생성은 하지 않습니다.

요약 경로에는 모든 상담에 같은 요약 기준을 사용하고, 질의응답 경로에는 미리 준비한 질문을 함께 전달합니다. 원문은 분석의 근거, 분류 결과는 처리 방향을 정하는 정보입니다.


In [29]:
question_specs = {
    40001: ("의문사형", "고객이 삼성 케어 플러스 보험 상품 가입 시 매장 코드를 어떻게 확인할 수 있나요?"),
    40003: ("의문사형", "고객의 결합 할인은 누구 앞으로 되어 있나요?"),
    40005: ("예/아니요형", "고객은 지금 추가 결합 할인을 받겠다고 했어?"),
    40006: ("예/아니요형", "고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?"),
    40007: ("예/아니요형", "상담사는 고객에게 가까운 대리점을 문자 메시지로 안내했습니까?"),
}
document_rows = []
for sid, (qa_type, question) in question_specs.items():
    cr = get_record(classification, sid, "상담 요건")
    sr = get_record(summarization, sid, "길이 제한 요약")
    qr = get_record(qa, sid, qa_type, question)
    if not (cr["input"] == sr["input"] == qr["input"]):
        raise ValueError("선택한 세 작업의 원문이 다릅니다. 원본부터 확인하세요.")
    document_rows.append({
        "document_id": sid, "source_id": sid, "text": cr["input"],
        "class_record_id": cr["record_id"], "summary_record_id": sr["record_id"],
        "qa_record_id": qr["record_id"], "question": qr["instruction"],
        "분류_기존지시": cr["instruction"], "요약_기존지시": sr["instruction"],
        "기존_category": cr["output"], "기존_summary": sr["output"], "기존_answer": qr["output"]
    })
documents = pd.DataFrame(document_rows)
print("상담별 입력 행 수:", len(documents))
print("문서 ID 고유:", documents["document_id"].is_unique)
print(documents[["document_id", "기존_category", "question"]].to_string(index=False))

ALLOWED_CATEGORIES = ["단일 요건 민원", "다수 요건 민원"]
CLASS_INSTRUCTION = (
    "상담 요건을 단일 요건 민원 또는 다수 요건 민원 중 하나로 분류하세요. "
    "중심 요청·주제가 하나면 단일, 구분되는 요청·주제가 둘 이상이면 다수입니다. "
    "한 요청의 절차를 여러 번 질문한 것만으로 다수로 보지 마세요. 범주명만 출력하세요."
)
SUMMARY_INSTRUCTION = (
    "제공된 상담 요건 분류를 참고하여 원문의 핵심 문의와 처리 상황을 요약하세요. "
    "여러 주제가 있으면 주제를 구분해 반영하고, 실제 완료와 추후 예정 사항을 구분하세요. "
    "원문에 없는 사실은 추가하지 마세요."
)

def get_document(sid):
    selected = documents[documents["document_id"] == sid]
    if len(selected) != 1:
        raise ValueError("문서 ID로 한 행을 선택해야 합니다.")
    return selected.iloc[0]

def make_follow_input(doc, category, route):
    if route == "상담 요약":
        instruction = SUMMARY_INSTRUCTION
    elif route == "질의응답":
        instruction = "상담 원문을 근거로 다음 질문에 답하세요. 질문: " + doc["question"]
    else:
        raise ValueError("정의되지 않은 후속 작업입니다.")
    return {"document_id": int(doc["document_id"]), "text": doc["text"],
            "category": category, "instruction": instruction}


상담별 입력 행 수: 5
문서 ID 고유: True
 document_id 기존_category                                       question
       40001    다수 요건 민원 고객이 삼성 케어 플러스 보험 상품 가입 시 매장 코드를 어떻게 확인할 수 있나요?
       40003    다수 요건 민원                      고객의 결합 할인은 누구 앞으로 되어 있나요?
       40005    다수 요건 민원                      고객은 지금 추가 결합 할인을 받겠다고 했어?
       40006    단일 요건 민원           고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?
       40007    다수 요건 민원             상담사는 고객에게 가까운 대리점을 문자 메시지로 안내했습니까?


---
## 필수 1. 단일 호출과 순차 처리 비교

### 문제 1-1. 분류·요약을 한 번에 얻는 방식과 단계로 나누는 방식

#### 문제 설명

상담 **40005**의 요건을 분류하고 핵심 내용을 요약하려고 합니다. 아래 두 설계를 비교합니다.

| 설계 | 처리 흐름 | 가정 |
|---|---|---|
| A: 단일 호출 | 원문을 넣어 category·summary를 함께 받음 | 두 결과를 별도 중간 처리 없이 한 번에 생성할 수 있음 |
| B: 순차 처리 | 원문으로 category 생성 → 원문+category로 summary 생성 → 결과 구성 | 분류 중간 결과를 확인하고 후속 요약에서 활용해야 함 |

제공 셀의 두 응답은 같은 내용을 담도록 작성한 가상 값입니다. 어느 방식의 실제 품질이 더 좋은지 측정하는 실험은 아닙니다.

#### 요구사항

1. `doc = get_document(40005)`로 문서를 선택하고 원문, 기존 분류·요약 지시와 출력을 확인하세요.
2. 단일 호출의 입력을 `single_input`에 구성하세요. document_id·text·instruction을 넣고, 지시는 분류와 요약을 함께 요구하도록 작성합니다. 기존 output을 입력에 넣지 않습니다.
3. 순차 처리의 분류 입력 `class_input`을 작성하고, 제공된 가상 `class_demo`가 허용 범주인지 확인하세요.
4. 허용되는 경우 `make_follow_input(doc, class_demo, "상담 요약")`으로 `summary_input`을 구성하세요. 원문과 분류 결과가 모두 전달되었는지 확인합니다.
5. 순차 처리의 단계별 `단계`, `입력 정보`, `출력 정보`, `LLM 요청 단계`를 담은 `stage_plan` 표를 작성하세요. 결과를 표에 모으는 단계도 표시하되 LLM 호출로 세지 않습니다.
6. `방식`, `document_id`, `원문`, `기존_category`, `기존_summary`, `새_category`, `새_summary`, `정상경로_예상호출수`를 포함한 `comparison` 표를 만드세요. A·B를 각각 한 행으로 작성합니다.

#### 해석 질문

**Q1.** 분류와 요약 두 결과가 필요하면 LLM을 반드시 두 번 호출해야 하나요? A를 우선 고려할 수 있는 조건을 설명하세요.

**Q2.** B의 작업 의존성은 어디에 있나요? 분류와 요약 코드를 단순히 앞뒤로 실행하는 것만으로 앞 결과를 활용한다고 할 수 있나요?

**Q3.** 요약 입력에서 원문을 빼고 분류명만 보내면 어떤 문제가 생기나요? 이번 원문에서 요약해야 할 실제 사실을 두 가지 적으세요.

**Q4.** B의 장점과 시간·비용·오류 전파 측면의 고려사항은 무엇인가요? 제공한 두 결과가 같다는 이유로 실제 품질이 같다고 결론 내릴 수 있나요?

#### 제출 결과

- 단일·분류·요약 단계의 입력 구성 코드와 전달 확인 결과
- stage_plan, comparison
- Q1~Q4 답변


### 제공 자료

아래 셀은 수정하지 않고 실행하세요.


### 준비 코드셀 함수 정리
**get_record(table, source_id, task_category, instruction=None)**
CSV 표에서 상담 ID와 작업 유형이 맞는 행을 찾는 함수다. 질의응답처럼 한 상담에 질문이 여러 개 있으면 instruction(질문)까지 넣어서 더 좁힌다. 찾은 행이 딱 1개가 아니면 ValueError를 내서 멈춘다. 행이 여러 개 걸리거나 하나도 없는데 그냥 넘어가 엉뚱한 행을 쓰는 일을 막기 위해서다. 결과는 행 하나(Series)로 돌려준다.

**show_record(row, include_input=False)**
get_record로 고른 행의 내용을 출력해서 확인하는 함수다. record_id, source_id, 작업 유형, 지시 또는 질문, CSV에 저장된 출력을 보여 준다. 원문은 길어서 기본으로는 출력하지 않고, include_input=True로 줄 때만 같이 출력한다.

**get_document(sid)**
documents 표에서 document_id가 sid인 상담 한 행을 꺼내는 함수다. documents는 상담별로 분류, 요약, 질의응답 행을 하나씩 골라 원문과 기존 출력, 지정 질문을 한 행에 모아 둔 표다. 여기서도 결과가 1행이 아니면 오류를 낸다. 필수 1에서는 get_document(40005), 필수 2에서는 get_document(40006)처럼 문서를 고를 때 쓴다.

**make_follow_input(doc, category, route)**
앞 단계의 분류 결과를 다음 단계 입력으로 넘길 때 쓰는 함수다. route가 "상담 요약"이면 SUMMARY_INSTRUCTION을 지시로 쓰고, "질의응답"이면 "상담 원문을 근거로 다음 질문에 답하세요. 질문: ..." 뒤에 그 문서의 지정 질문을 붙여 지시로 쓴다. 그 외의 경로가 들어오면 정의되지 않은 작업이라며 오류를 낸다. 반환값은 document_id, text(원문), category(분류 결과), instruction(지시)을 담은 딕셔너리다. 입력을 묶어 주기만 할 뿐 실제로 API를 호출하거나 응답을 만들지는 않는다.

**함수에서 같이 쓰는 상수**
- ALLOWED_CATEGORIES는 허용하는 분류 값 목록으로 "단일 요건 민원"과 "다수 요건 민원" 두 가지다. 분류 결과가 형식에 맞는지 검사할 때 쓴다.
- CLASS_INSTRUCTION은 분류 단계의 지시다. 중심 요청이 하나면 단일, 둘 이상이면 다수로 나누고 범주명만 출력하라고 한다.
- SUMMARY_INSTRUCTION은 요약 단계의 지시다. 분류 결과를 참고해서 주제를 나눠 요약하고, 실제로 완료된 일과 나중에 할 예정인 일을 구분하고, 원문에 없는 사실은 넣지 말라고 한다. make_follow_input에서 요약 경로를 고르면 이 지시가 들어간다.

In [30]:
# 제공 자료: 실제 LLM 응답이 아닌 가상 값입니다.
demo_summary_40005 = (
    "통화 중 대기 서비스를 재설정하고 시험 통화로 정상 작동을 확인했다. "
    "추가 결합 할인은 즉시 연결하지 않고 추후 상담 연락을 받기로 했다."
)
single_demo = {"category": "다수 요건 민원", "summary": demo_summary_40005}
class_demo = "다수 요건 민원"
summary_demo = demo_summary_40005
# 싱글 데모와 요약 데모 값을 같게 하여 출력이 같도록 설정 필수 1 코드 출력 내용 보면 같은 것을 확인할 수 있음(실제 llm 응답은 거의 다를 것임.)


In [31]:
# 문제 1 풀이 코드를 작성하세요.

# 1. 문서 선택과 원문, 기존 지시, 기존 출력 확인
doc = get_document(40005)
print("[상담 40005 원문]")
# print(doc["text"])
print("\n분류 기존 지시:", doc["분류_기존지시"])
print("요약 기존 지시:", doc["요약_기존지시"])
print("기존_category:", doc["기존_category"])
print("기존_summary:", doc["기존_summary"])

# 2. A: 단일 호출 입력 (분류와 요약을 한 번에 요구, 기존 output은 넣지 않음)
single_input = {
    "document_id": int(doc["document_id"]),
    "text": doc["text"],
    "instruction": (
        CLASS_INSTRUCTION.replace(" 범주명만 출력하세요.", "")
        + " 그리고 원문의 핵심 문의와 처리 상황을 요약하세요. 실제 완료와 추후 예정 사항을 구분하고 "
        "원문에 없는 사실은 추가하지 마세요. 결과는 category(범주명)와 summary(요약) 두 항목으로 출력하세요."
    ),
}
print("\n[single_input]")
print("항목:", list(single_input.keys()))
print("instruction:", single_input["instruction"])
print("원문 전체 전달:", single_input["text"] == doc["text"])
print("기존 출력 항목이 입력에 없음:", not any(k == "output" or k.startswith("기존") for k in single_input))
print("기존 요약 문장이 지시에 없음:", doc["기존_summary"] not in single_input["instruction"])

# 3. B-1단계: 분류 입력과 가상 분류 결과 확인
class_input = {
    "document_id": int(doc["document_id"]),
    "text": doc["text"],
    "instruction": CLASS_INSTRUCTION,
}
print("\n[class_input] instruction:", class_input["instruction"])
print("가상 분류 결과:", class_demo)
print("허용 범주 여부:", class_demo in ALLOWED_CATEGORIES)

# 4. B-2단계: 허용되는 경우에만 원문과 분류 결과를 요약 입력으로 전달
if class_demo in ALLOWED_CATEGORIES:
    summary_input = make_follow_input(doc, class_demo, "상담 요약")
    print("\n[summary_input]")
    print("항목:", list(summary_input.keys()))
    print("원문 전달:", summary_input["text"] == doc["text"])
    print("분류 결과 전달:", summary_input["category"] == class_demo, "/ 값:", summary_input["category"])
    print("요약 지시:", summary_input["instruction"])
else:
    summary_input = None
    print("허용되지 않은 분류라 요약 단계를 실행하지 않습니다.")

# 5. 순차 처리 단계 계획
stage_plan = pd.DataFrame([
    {"단계": "1. 분류", "입력 정보": "원문 + 분류 지시", "출력 정보": "category", "LLM 요청 단계": True},
    {"단계": "2. 분류 결과 확인", "입력 정보": "category", "출력 정보": "허용 범주 여부(진행/중단)", "LLM 요청 단계": False},
    {"단계": "3. 요약", "입력 정보": "원문 + category + 요약 지시", "출력 정보": "summary", "LLM 요청 단계": True},
    {"단계": "4. 결과 구성", "입력 정보": "category + summary + 문서 ID", "출력 정보": "comparison 표의 한 행", "LLM 요청 단계": False},
])
print("\n[stage_plan]")
print(stage_plan.to_string(index=False))
print("순차 처리의 LLM 요청 단계 수:", stage_plan["LLM 요청 단계"].sum())

# 6. A와 B 비교표 (가상 결과, 정상 경로 기준 예상 호출 수)
comparison = pd.DataFrame([
    {"방식": "A: 단일 호출", "document_id": int(doc["document_id"]), "원문": doc["text"],
     "기존_category": doc["기존_category"], "기존_summary": doc["기존_summary"],
     "새_category": single_demo["category"], "새_summary": single_demo["summary"],
     "정상경로_예상호출수": 1},
    {"방식": "B: 순차 처리", "document_id": int(doc["document_id"]), "원문": doc["text"],
     "기존_category": doc["기존_category"], "기존_summary": doc["기존_summary"],
     "새_category": class_demo, "새_summary": summary_demo,
     "정상경로_예상호출수": int(stage_plan["LLM 요청 단계"].sum())},
])
print("\n[comparison] (원문은 글자 수만 표시)")
print(comparison.assign(원문=comparison["원문"].str.len().astype(str) + "자").to_string(index=False))
print("\n두 방식의 가상 결과가 같음:",
      comparison.loc[0, "새_category"] == comparison.loc[1, "새_category"]
      and comparison.loc[0, "새_summary"] == comparison.loc[1, "새_summary"])

# 참고: 요약해야 할 실제 사실이 원문에 있는지 확인
for p in ["통화 중 대기 서비스 삭제를 한 번 했고요", "다시 한번 신청해봐 드릴게요",
          "다음에 할게요", "네, 그렇게 해주세요"]:
    print(f"원문에 '{p}' 존재:", p in doc["text"])


[상담 40005 원문]

분류 기존 지시: 이 상담의 내용이 주제가 하나인 "단일 요건 민원"인지, 여러 개의 주제가 있는 "다수 요건 민원"인지 알려 줘.
요약 기존 지시: LGU+ 통화 중 대기 서비스에 대한 상담 내용을 2문장, 300자 이내로 줄여 줘.
기존_category: 다수 요건 민원
기존_summary: 고객은 통화 중 대기 서비스의 작동 방식에 대해 문의하였고, 상담사는 서비스 삭제 후 재설정하여 정상 작동을 확인해주었다. 추가로 LGU+ 결합 할인 혜택에 대해 안내하였으나 고객은 추후 검토하기로 하였다.

[single_input]
항목: ['document_id', 'text', 'instruction']
instruction: 상담 요건을 단일 요건 민원 또는 다수 요건 민원 중 하나로 분류하세요. 중심 요청·주제가 하나면 단일, 구분되는 요청·주제가 둘 이상이면 다수입니다. 한 요청의 절차를 여러 번 질문한 것만으로 다수로 보지 마세요. 그리고 원문의 핵심 문의와 처리 상황을 요약하세요. 실제 완료와 추후 예정 사항을 구분하고 원문에 없는 사실은 추가하지 마세요. 결과는 category(범주명)와 summary(요약) 두 항목으로 출력하세요.
원문 전체 전달: True
기존 출력 항목이 입력에 없음: True
기존 요약 문장이 지시에 없음: True

[class_input] instruction: 상담 요건을 단일 요건 민원 또는 다수 요건 민원 중 하나로 분류하세요. 중심 요청·주제가 하나면 단일, 구분되는 요청·주제가 둘 이상이면 다수입니다. 한 요청의 절차를 여러 번 질문한 것만으로 다수로 보지 마세요. 범주명만 출력하세요.
가상 분류 결과: 다수 요건 민원
허용 범주 여부: True

[summary_input]
항목: ['document_id', 'text', 'category', 'instruction']
원문 전달: True
분류 결과 전달: True / 값: 다수 요건 민원
요약 지시: 제공된 상담 요건 분류를 참고하

### 해석 답안 작성

- Q1. 꼭 두 번 호출해야 하는 건 아니다. A처럼 원문을 한 번 넣고 category와 summary를 함께 받을 수도 있고, 이번 `single_input`도 지시 하나에 분류와 요약을 같이 요구했다. 두 결과가 서로 기다릴 필요 없이 원문만 있으면 바로 만들 수 있는 경우, 즉 요약을 만들 때 분류 결과를 먼저 확인해서 써야 할 이유가 없을 때는 A를 먼저 고려할 수 있다. 또 중간 결과를 따로 검사하거나 분류에 따라 다음 작업을 바꿀 필요가 없고, 호출 수와 처리 시간을 줄이는 게 중요할 때도 A가 적합하다.
-> 반드시 주 번 호출할 필요는 없다. 중간 결과를 따로 확인하거나 후속 입력에 전달할 필요가 없고, 한 요청으로 분류, 요약을 충분히 생성할 수 있다면 A를 우선 고려할 수 있다.

- Q2. B의 의존성은 요약 단계가 분류 단계의 결과(category)를 입력으로 받는다는 데 있다. 실제로 `summary_input`에 원문과 함께 category "다수 요건 민원"이 들어갔고(분류 결과 전달 True), 요약 지시도 "제공된 상담 요건 분류를 참고하여"라고 되어 있다. 분류 코드와 요약 코드를 그냥 앞뒤로 실행하기만 하고 분류 결과를 요약 입력에 넣지 않으면, 순서만 있을 뿐 앞 결과를 활용했다고 할 수 없다. 앞 단계의 출력이 확인을 거쳐 다음 단계 입력에 실제로 들어가야 순차 처리의 의미가 있다.
-> B의 의존성은 첫 단계의 카테고리가 다음 요약 입력에 들어가는 부분이다. 단순히 분류 코드 다음에 요약 코드를 실행해도 요약이 분류 결과를 받지 않는다면, 앞 결과를 활용하는 연결을 구현한 것은 아니다.
-> 실제로 전달하는 값과 후속지시에서의 역할을 함께 확인해야 한다.

- Q3. 분류명 "다수 요건 민원"은 주제가 여러 개라는 것만 알려 주고 무슨 일이 있었는지는 담고 있지 않다. 원문 없이 분류명만 보내면 요약할 사실이 없어서 그럴듯한 내용을 지어내거나 아무 의미 없는 요약이 나올 수 있다. 이번 원문에서 요약해야 할 실제 사실은 첫째, 상담사가 통화 중 대기 서비스를 삭제한 뒤 다시 신청하고 테스트 전화로 정상 작동을 확인한 것("통화 중 대기 서비스 삭제를 한 번 했고요", "다시 한번 신청해봐 드릴게요")이다. 둘째, 추가 결합 할인은 고객이 "다음에 할게요"라고 해서 지금 연결하지 않았고, 추후 상담사 연락에만 "네, 그렇게 해주세요"라고 동의한 것이다.
-> 다수 요건 민원이라는 분류명만으로는 어떤 서비스 문제와 요청이 있었는지 알 수 없다.
-> 원문이 없으면 구체적 사실을 근거로 요약하기 어렵다.
-> 이 상담에서는 통화 중 대기 서비스의 재설정, 정상 확인과 추가 할인 상담의 추후 연락 동의를 구분해 반영해야 한다.

- Q4. B의 장점은 분류 결과를 중간에 확인할 수 있어서 허용 범주가 아니면 요약 전에 멈출 수 있고, 분류 결과를 요약에 활용할 수 있고, 어느 단계에서 문제가 생겼는지 나눠서 볼 수 있다는 점이다. 대신 정상 경로 기준으로 호출이 A는 1번, B는 2번이라서 처리 시간과 사용량이 늘어날 수 있고, 원문을 두 번 보내니 입력 토큰도 더 든다. 또 분류가 틀리면 그 틀린 분류가 요약 입력으로 넘어가서 요약까지 영향을 받는 오류 전파가 생길 수 있다. 이번 두 결과는 같은 내용으로 작성한 가상 값이라서(가상 결과 같음 True) 실제 품질이 같다고 결론 낼 수 없다. 실제로 비교하려면 두 방식을 실제로 실행해서 같은 문서들과 같은 기준으로 평가해야 한다.
-> B는 중간 분류를 따로 확인하고 특정 단계의 결과를 점검하기 쉽다.
-> 여러 요청으로 처리 시간, 사용량, 비용이 늘 수 있고, 잘못된 분류를 다음 요약에 전달하면 그 오류가 영향을 줄 수 있다.
-> 호출이 두 번이라고 반드시 비용이 두 배라고 단정지을 수 없다.
-> 마찬가지로 입력, 출력 규모 등이 다르기 때문에 실제 품질의 동일성을 증명하지 않는다.


---
## 필수 2. 조건부 분기와 후속 입력 연결

### 문제 2-1. 상담 요건에 따라 요약 또는 질의응답 선택하기

#### 문제 설명

이번 분석기의 목표는 **여러 주제가 있는 상담은 전체 문의와 처리 상황을 요약하고, 하나의 중심 요청을 가진 상담은 미리 정한 확인 질문에 답하는 것**입니다. 다음 경로를 개발자가 미리 정했다고 가정합니다.

| 분류 결과 | 선택할 경로 | 후속 입력 |
|---|---|---|
| 다수 요건 민원 | 상담 요약 | 원문 + 분류 결과 + 요약 지시 |
| 단일 요건 민원 | 질의응답 | 원문 + 분류 결과 + 지정 질문을 포함한 지시 |
| 그 외 값·결과 없음 | 경로 선택 안 함 | 오류를 확인한 뒤 후속 처리 여부 결정 |

이것은 **이번 분석 목적에 맞춘 연습 규칙**입니다. 모든 단일 상담은 질의응답만 해야 한다거나 다수 상담에는 질의응답을 할 수 없다는 뜻이 아닙니다.

상담 40005·40006에 대해 가상 분류와 후속 결과가 제공됩니다. 40006의 지정 질문은 `고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?`입니다.

#### 요구사항

1. `get_document(40006)`으로 원문과 지정 질문을 확인하세요. 40005 원문은 필수 1에서 확인한 내용을 사용합니다.
2. 위 조건에 따라 `"상담 요약"`, `"질의응답"`, `None` 중 하나를 반환하는 `choose_route(category)` 함수를 if·elif·else로 작성하세요.
3. 두 상담을 차례로 처리하면서 **제공된 가상 분류 결과**로 경로를 선택하세요. CSV의 기존_category를 분기 입력 대신 사용하지 않습니다.
4. 선택된 경로에 대해 `make_follow_input()`으로 후속 입력을 만들고 `follow_inputs` 목록에 보관하세요. ID·원문·분류 결과가 원래 값과 일치하는지 확인하고 지시를 출력합니다.
5. 선택된 경로의 가상 출력만 가져와 `branch_results`를 만드세요. document_id·text·기존 출력 3개·새 category·route·final_result를 보존합니다.
6. 별도 점검으로 `choose_route("복합 문의")`, `choose_route(None)`을 실행하세요. 이어서 40005가 잘못 `단일 요건 민원`으로 분류되었다고 가정할 때 선택될 경로를 확인하고, 원문에 맞는 분류인지 별도로 설명하세요. 이 점검 값으로 정상 결과를 덮어쓰지 않습니다.

#### 해석 질문

**Q1.** 각 상담은 어떤 경로로 가며 다음 단계에 어떤 정보가 필요한가요? 선택되지 않은 경로까지 모두 실행해야 하나요?

**Q2.** 분류·후속 분석 순서가 있다는 점에서 순차 처리와 비슷하지만, 이 설계에서 조건부 분기라고 부르는 이유는 무엇인가요?

**Q3.** `복합 문의`와 `단일 요건 민원`으로 잘못 분류된 40005는 무엇이 다른가요? 형식 오류와 오류 전파를 구분하여 설명하세요.

**Q4.** 이 워크플로우를 Agent의 자율적 작업 선택이라고 볼 수 있나요? LLM 분류 결과 외에 무엇을 기준으로 분기할 수 있나요?

#### 제출 결과

- choose_route 함수, 두 상담의 후속 입력과 전달 확인
- branch_results, 별도 경로 점검 결과
- Q1~Q4 답변


### 제공 자료

아래 셀은 수정하지 않고 실행하세요.


### 준비셀 함수 정리
**get_record(table, source_id, task_category, instruction=None)**
CSV 표에서 상담 ID와 작업 유형이 맞는 행을 찾는 함수다. 질의응답처럼 한 상담에 질문이 여러 개 있으면 instruction(질문)까지 넣어서 더 좁힌다. 찾은 행이 딱 1개가 아니면 ValueError를 내서 멈춘다. 행이 여러 개 걸리거나 하나도 없는데 그냥 넘어가 엉뚱한 행을 쓰는 일을 막기 위해서다. 결과는 행 하나(Series)로 돌려준다.

**show_record(row, include_input=False)**
get_record로 고른 행의 내용을 출력해서 확인하는 함수다. record_id, source_id, 작업 유형, 지시 또는 질문, CSV에 저장된 출력을 보여 준다. 원문은 길어서 기본으로는 출력하지 않고, include_input=True로 줄 때만 같이 출력한다.

**get_document(sid)**
documents 표에서 document_id가 sid인 상담 한 행을 꺼내는 함수다. documents는 상담별로 분류, 요약, 질의응답 행을 하나씩 골라 원문과 기존 출력, 지정 질문을 한 행에 모아 둔 표다. 여기서도 결과가 1행이 아니면 오류를 낸다. 필수 1에서는 get_document(40005), 필수 2에서는 get_document(40006)처럼 문서를 고를 때 쓴다.

**make_follow_input(doc, category, route)**
앞 단계의 분류 결과를 다음 단계 입력으로 넘길 때 쓰는 함수다. route가 "상담 요약"이면 SUMMARY_INSTRUCTION을 지시로 쓰고, "질의응답"이면 "상담 원문을 근거로 다음 질문에 답하세요. 질문: ..." 뒤에 그 문서의 지정 질문을 붙여 지시로 쓴다. 그 외의 경로가 들어오면 정의되지 않은 작업이라며 오류를 낸다. 반환값은 document_id, text(원문), category(분류 결과), instruction(지시)을 담은 딕셔너리다. 입력을 묶어 주기만 할 뿐 실제로 API를 호출하거나 응답을 만들지는 않는다.

**함수에서 같이 쓰는 상수**
- ALLOWED_CATEGORIES는 허용하는 분류 값 목록으로 "단일 요건 민원"과 "다수 요건 민원" 두 가지다. 분류 결과가 형식에 맞는지 검사할 때 쓴다.
- CLASS_INSTRUCTION은 분류 단계의 지시다. 중심 요청이 하나면 단일, 둘 이상이면 다수로 나누고 범주명만 출력하라고 한다.
- SUMMARY_INSTRUCTION은 요약 단계의 지시다. 분류 결과를 참고해서 주제를 나눠 요약하고, 실제로 완료된 일과 나중에 할 예정인 일을 구분하고, 원문에 없는 사실은 넣지 말라고 한다. make_follow_input에서 요약 경로를 고르면 이 지시가 들어간다.

In [32]:
# 제공 자료: 두 상담의 가상 중간 결과와 가상 후속 응답입니다.
branch_class_demo = {40005: "다수 요건 민원", 40006: "단일 요건 민원"}
branch_follow_demo = {
    (40005, "상담 요약"): "통화 중 대기 재설정 후 정상 작동을 확인했다. 추가 결합 할인은 추후 상담 연락을 받기로 했다.",
    (40006, "질의응답"): "네. 상담사는 매장을 방문해 해지할 때 명의자 신분증을 지참하라고 안내했다."
}

In [33]:
# 문제 2 풀이 코드를 작성하세요.

# 1. 40006 원문과 지정 질문 확인 (40005 원문은 필수 1에서 확인)
doc_40006 = get_document(40006)
# print("[상담 40006 원문]")
# print(doc_40006["text"])
print("\n지정 질문:", doc_40006["question"])

# 2. 미리 정한 규칙으로 경로 선택
def choose_route(category):
    if category == "다수 요건 민원":
        return "상담 요약"
    elif category == "단일 요건 민원":
        return "질의응답"
    else:
        return None

# 3, 4, 5. 두 상담을 차례로 처리 (가상 분류 결과로 분기, 기존_category는 쓰지 않음)
follow_inputs = []
branch_rows = []
for sid in [40005, 40006]:
    d = get_document(sid)
    category = branch_class_demo[sid]
    route = choose_route(category)
    print(f"\n===== 상담 {sid} =====")
    print("가상 분류 결과:", category, "/ 선택 경로:", route)
    if route is None:
        print("경로가 선택되지 않아 후속 입력을 만들지 않습니다.")
        continue

    follow_input = make_follow_input(d, category, route)
    follow_inputs.append(follow_input)
    print("ID 일치:", follow_input["document_id"] == sid)
    print("원문 일치:", follow_input["text"] == d["text"])
    print("분류 결과 일치:", follow_input["category"] == category)
    print("후속 지시:", follow_input["instruction"])

    final_result = branch_follow_demo[(sid, route)]   # 선택된 경로의 가상 출력만 가져옴
    branch_rows.append({
        "document_id": sid, "text": d["text"],
        "기존_category": d["기존_category"], "기존_summary": d["기존_summary"], "기존_answer": d["기존_answer"],
        "category": category, "route": route, "final_result": final_result,
    })

branch_results = pd.DataFrame(branch_rows)
print("\n[branch_results] (원문은 글자 수만 표시)")
print(branch_results.assign(text=branch_results["text"].str.len().astype(str) + "자").to_string(index=False))
print("후속 입력 수:", len(follow_inputs))



지정 질문: 고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?

===== 상담 40005 =====
가상 분류 결과: 다수 요건 민원 / 선택 경로: 상담 요약
ID 일치: True
원문 일치: True
분류 결과 일치: True
후속 지시: 제공된 상담 요건 분류를 참고하여 원문의 핵심 문의와 처리 상황을 요약하세요. 여러 주제가 있으면 주제를 구분해 반영하고, 실제 완료와 추후 예정 사항을 구분하세요. 원문에 없는 사실은 추가하지 마세요.

===== 상담 40006 =====
가상 분류 결과: 단일 요건 민원 / 선택 경로: 질의응답
ID 일치: True
원문 일치: True
분류 결과 일치: True
후속 지시: 상담 원문을 근거로 다음 질문에 답하세요. 질문: 고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?

[branch_results] (원문은 글자 수만 표시)
 document_id  text 기존_category                                                                                                          기존_summary 기존_answer category route                                           final_result
       40005 2297자    다수 요건 민원 고객은 통화 중 대기 서비스의 작동 방식에 대해 문의하였고, 상담사는 서비스 삭제 후 재설정하여 정상 작동을 확인해주었다. 추가로 LGU+ 결합 할인 혜택에 대해 안내하였으나 고객은 추후 검토하기로 하였다.       아니. 다수 요건 민원 상담 요약 통화 중 대기 재설정 후 정상 작동을 확인했다. 추가 결합 할인은 추후 상담 연락을 받기로 했다.
       40006 2463자    단일 요건 민원                       고객은 LGU+ 휴대폰 해지를 위해 대리점이나 직영점 방문을

### 해석 답안 작성

- Q1. 40005는 가상 분류가 "다수 요건 민원"이라서 상담 요약 경로로 가고, 다음 단계에는 원문, 분류 결과, 요약 지시가 필요하다. 40006은 "단일 요건 민원"이라서 질의응답 경로로 가고, 원문, 분류 결과, 그리고 지정 질문("고객이 대리점을 방문하여 해지 신청을 할 때 신분증이 필요합니까?")이 들어간 지시가 필요하다. 두 경우 모두 ID, 원문, 분류 결과가 원래 값과 일치했다. 선택되지 않은 경로까지 실행할 필요는 없다. 조건부 분기는 조건에 맞는 경로 하나만 실행해서 불필요한 호출을 줄이는 방식이고, 그래서 `follow_inputs`에도 상담마다 하나씩, 총 2개만 만들었다.
-> 40005는 상담 요약, 40006은 질의 응답 경로로 간다.
-> 두 경로 모두 문서 아이디, 원문, 분류 결과, 해당 작업의 지시가 필요하고 질의응답에는 지정 질문이 포함된다.
-> 조건에 따라 선택된 한 경로만 수행한다. 두 경로를 모두 싫행하면 이번 분기 설계가 달라지고 불필요한 작업이 늘 수 있다.

- Q2. 분류 다음에 후속 분석이 오는 순서가 있다는 점은 순차 처리와 같다. 하지만 순차 처리는 모든 문서가 같은 다음 단계로 가는 반면, 이 설계는 분류 결과가 무엇이냐에 따라 다음에 할 작업 자체가 요약 또는 질의응답으로 달라진다. 실제로 같은 코드를 돌렸는데 40005는 상담 요약, 40006은 질의응답으로 갈라졌다. 앞 결과가 다음 입력으로 쓰이는 것뿐 아니라 어떤 작업을 할지 고르는 조건으로 쓰이기 때문에 조건부 분기라고 부른다.
-> 순차 처리에서는 같은 후속 작업을 정해진 순서로 실행하며 앞 결과를 입력으로 쓴다.
-> 이번 설계는 분류 결과에 따라 요약 또는 질의응답이라는 작업 자체가 달라진다.
-> 단계들이 순서대로 연결되어 있으면서 그 안에 조건부 분기가 있는 형태다.

- Q3. "복합 문의"는 허용 범주에 없는 값이라서 `choose_route()`가 None을 반환하고 경로가 선택되지 않는다. 이건 형식 오류라서 코드로 바로 잡아낼 수 있고, 후속 단계로 넘어가기 전에 멈출 수 있다. 반면 40005를 "단일 요건 민원"으로 잘못 분류한 경우는 허용 범주라서 형식 검사를 통과하고, 그대로 질의응답 경로가 선택된다. 하지만 40005는 통화 중 대기 문제와 추가 결합 할인 안내라는 두 주제가 있어서 원문상 다수 요건 민원이 맞다. 이렇게 틀린 중간 결과가 다음 단계로 넘어가서 원래 해야 할 전체 요약 대신 질의응답이 실행되는 게 오류 전파다. 형식 검사로는 막을 수 없어서 원문 기준 검토가 따로 필요하다.
-> 복합 문의는 허용 목록에 없는 값이라 출력 조건 위반이며 경로를 결정하지 않음
-> 단일 요건 민원은 허용돈 문자열이므로 40005에 잘못 부여되어도 형식 검사와 분기를 통과할 수 있다.
-> 이 경우 원문에 맞지 않는 중간 분류가 후속 작업 선택에 영향을 준다.
-> 형식 검사는 의미 오류까지 막아 주지 못하므로 원문과 판단 기준의 비교가 별도로 필요하다.

- Q4. Agent의 자율적 작업 선택이라고 보기 어렵다. 경로는 개발자가 if, elif, else로 미리 정한 규칙(다수면 요약, 단일이면 질의응답, 그 외는 None)에 따라 정해지고, 질문도 미리 지정한 것이다. LLM은 분류 값을 만들 뿐 어떤 작업을 할지 스스로 계획하거나 고르지 않는다. 분기 기준은 LLM 분류 외에도 원문 길이(예를 들어 일정 글자 수가 넘으면 요약), 원본 데이터의 상담 카테고리 같은 메타데이터, 특정 단어가 들어 있는지 같은 규칙, 앞 단계의 요청 상태(성공, 실패) 등을 쓸 수 있다.
-> Agent의 자율적 선택이 아니다. 개발자가 두 분기 조건과 수행 작업을 미리 정했고 프로그램이 그 규칙을 적용한다. LLM 분류 결과뿐 아니라 데이터의 작업 유형처럼 이미 있는 값이나 사전에 정한 규칙으로도 분기할 수 있다.
-> 이번 예제에서는 함수도 다른 외부 도구를 새로 선택하거나 절차를 스스로 만들어내는 작업은 포함하지 않았다.



---
## 과제 1. 단계별 검증과 실패 처리

### 문제 3-1. 다섯 상담의 워크플로우 상태 관리

#### 문제 설명

필수 2의 분기 규칙을 다섯 상담에 적용합니다. 아래 가상 응답 기록을 재현하면서, 어디에서 문제가 생겼는지 남기세요. 필수 2의 `choose_route()`를 먼저 완성하고 사용합니다.

| 문서 | 분류 요청 상태 | 가상 분류 출력 | 후속 상황 |
|---|---|---|---|
| 40001 | completed | 다수 요건 민원 | 요약 응답 정상 |
| 40003 | completed | 다수 요건 민원 | 요약 요청 실패 |
| 40005 | completed | 복합 문의 | 분류 검증에서 중단해야 함 |
| 40006 | completed | 단일 요건 민원 | 질의응답 응답 정상 |
| 40007 | failed | 없음 | 분류 요청 실패로 중단해야 함 |

completed·failed는 실습용 요청 상태 값입니다. 분류 요청이 completed라고 해도 출력 값이 허용 범주가 아니면 후속 분석으로 진행할 수 없습니다.

상태를 다음과 같이 구분합니다.

| 기록 | 값 | 뜻 |
|---|---|---|
| class_status | success / error / not_run | 분류 단계의 검증 통과 / 오류 / 미실행 |
| follow_status | success / error / not_run | 후속 단계의 검증 통과 / 오류 / 미실행 |
| status | success / error | 전체 워크플로우의 정상 완료 여부 |
| failed_stage | 입력 준비 / 분류 / 분기 / 후속 분석 / 빈 문자열 | 문제가 발견된 단계 |
| error_reason | 설명 문자열 | 실패하거나 중단한 원인 |

이 예시의 success는 처리 및 기본 출력 조건을 통과했다는 뜻입니다. 원문과의 의미 일치까지 자동 검증했다는 뜻이 아닙니다. 기록된 실패 원인은 실제 서비스에서 발생한 장애가 아닙니다.

#### 요구사항

1. documents의 다섯 행을 순회하며 결과 행을 하나씩 만드세요. 시작할 때 category·route·final_result는 None, 단계 상태는 not_run, 전체 status는 error로 둡니다.
2. 원문이 없거나 비어 있으면 `입력 준비` 오류로 기록하고 다음 문서로 넘어갑니다. 제공된 실제 다섯 원문은 모두 존재하므로 정상 경로에서는 이 오류가 발생하지 않습니다.
3. `class_events`에서 해당 문서의 분류 기록을 확인합니다. 요청 실패, 출력 없음, 허용 범주가 아닌 값은 `분류` 오류로 기록하고 후속 응답을 조회하지 마세요. `raw_category`에는 제공된 원래 출력을 남깁니다.
4. 유효한 분류이면 category에 저장하고 class_status를 success로 설정합니다. choose_route()로 경로를 고르며, 경로가 없으면 `분기` 오류로 기록합니다.
5. 경로가 있으면 make_follow_input()으로 후속 입력을 만들고 `workflow_inputs`에 보관하세요. 이어서 해당 문서 ID를 `follow_attempted_ids`에 기록하고 `follow_events`의 가상 후속 응답을 읽습니다.
6. 후속 요청이 실패했거나 결과가 없으면 `후속 분석` 오류로 기록하고 final_result는 None으로 유지하세요. 정상인 경우에만 follow_status·status를 success로 바꾸고 최종 결과를 저장합니다. 분류가 성공한 뒤 후속 단계가 실패해도 이미 얻은 category는 보존합니다.
7. `workflow_results`에 입력 문서 5개를 모두 한 행씩 남기세요. document_id·원문·기존 출력 외에 raw_category·category·route·class_status·follow_status·final_result·status·failed_stage·error_reason을 포함합니다.
8. 전체 성공 건수와 실패 단계별 건수를 집계하고, `review_needed`를 만드세요. `follow_attempted_ids`에 40005·40007이 들어가지 않았는지도 확인하세요. 실제 재요청은 수행하지 않습니다.

**보조 구문:** `for _, doc in documents.iterrows():`, `row = doc.to_dict()`, `row.update({...})`, `rows.append(row)`, `continue`. `continue`는 현재 문서의 처리를 끝내고 다음 문서로 넘어가는 문법입니다. 중단하기 전에 결과 행을 추가해야 해당 문서의 기록이 남습니다.

#### 해석 질문

**Q1.** 정상 완료 문서는 몇 개이며 어떤 ID인가요? 분류 단계 오류와 후속 분석 오류의 ID도 구분하세요.

**Q2.** 40005와 40007은 왜 후속 분석을 실행하지 않나요? 두 문서 모두 최종 결과가 없지만 원인은 어떻게 다른가요?

**Q3.** 40003을 다시 처리한다면 분류부터 반드시 다시 해야 하나요? 유지해야 할 정보와 재처리 전 확인할 사항을 적으세요.

**Q4.** 최종 결과가 있는 행만 남기거나, 실패 행의 category·final_result를 기존 출력으로 채우면 어떤 문제가 있나요? 전체 결과 연결 검증에는 무엇이 필요한가요?

#### 제출 결과

- 입력·중간 결과 검증 및 중단 코드
- workflow_results 5행, workflow_inputs, follow_attempted_ids
- 성공·실패 집계, review_needed, Q1~Q4 답변


### 제공 자료

아래 셀은 수정하지 않고 실행하세요.


In [34]:
# 제공 자료: 가상 응답 기록. 실제 API 요청을 실행하지 않습니다.
class_events = {
    40001: {"request_status": "completed", "output": "다수 요건 민원", "error": ""},
    40003: {"request_status": "completed", "output": "다수 요건 민원", "error": ""},
    40005: {"request_status": "completed", "output": "복합 문의", "error": ""},
    40006: {"request_status": "completed", "output": "단일 요건 민원", "error": ""},
    40007: {"request_status": "failed", "output": None, "error": "분류 요청 실패(가상)"},
}
follow_events = {
    40001: {"request_status": "completed", "output": "고객은 사은품 케이스, 보험 매장 코드, 카드 할인과 휴대폰 정지 횟수를 문의하고 관련 안내를 받았다.", "error": ""},
    40003: {"request_status": "failed", "output": None, "error": "요약 요청 실패(가상)"},
    40006: {"request_status": "completed", "output": "네. 상담사는 매장에서 해지할 때 명의자 신분증을 지참하라고 안내했다.", "error": ""},
}
# 40005·40007의 후속 응답은 없습니다. 앞 단계에서 중단해야 하는 사례입니다.
print("분류 기록 ID:", list(class_events))
print("후속 응답 기록 ID:", list(follow_events))


분류 기록 ID: [40001, 40003, 40005, 40006, 40007]
후속 응답 기록 ID: [40001, 40003, 40006]


In [35]:
# 문제 3 풀이 코드를 작성하세요.

workflow_inputs = []
follow_attempted_ids = []
rows = []

for _, doc in documents.iterrows():
    sid = int(doc["document_id"])
    row = doc.to_dict()
    # 시작 상태: 결과는 None, 단계는 not_run, 전체는 error
    row.update({"raw_category": None, "category": None, "route": None,
                "class_status": "not_run", "follow_status": "not_run",
                "final_result": None, "status": "error", "failed_stage": "", "error_reason": ""})

    # 1) 입력 준비 확인
    text = doc["text"]
    if not isinstance(text, str) or not text.strip():
        row.update({"failed_stage": "입력 준비", "error_reason": "원문이 없거나 비어 있음"})
        rows.append(row)
        continue

    # 2) 분류 단계 확인
    event = class_events.get(sid)
    if event is None:
        row.update({"class_status": "error", "failed_stage": "분류", "error_reason": "분류 기록 없음"})
        rows.append(row)
        continue
    row["raw_category"] = event["output"]          # 원래 출력은 그대로 남김
    if event["request_status"] != "completed":
        row.update({"class_status": "error", "failed_stage": "분류",
                    "error_reason": "분류 요청 실패: " + event["error"]})
        rows.append(row)
        continue
    if event["output"] is None or not str(event["output"]).strip():
        row.update({"class_status": "error", "failed_stage": "분류", "error_reason": "분류 출력 없음"})
        rows.append(row)
        continue
    if event["output"] not in ALLOWED_CATEGORIES:
        row.update({"class_status": "error", "failed_stage": "분류",
                    "error_reason": "허용되지 않은 분류 값: " + event["output"]})
        rows.append(row)
        continue
    row.update({"category": event["output"], "class_status": "success"})

    # 3) 분기
    route = choose_route(row["category"])
    if route is None:
        row.update({"failed_stage": "분기", "error_reason": "정의된 경로가 없음"})
        rows.append(row)
        continue
    row["route"] = route

    # 4) 후속 분석
    follow_input = make_follow_input(doc, row["category"], route)
    workflow_inputs.append(follow_input)
    follow_attempted_ids.append(sid)
    f_event = follow_events.get(sid)
    if f_event is None or f_event["request_status"] != "completed":
        reason = "후속 응답 기록 없음" if f_event is None else "후속 요청 실패: " + f_event["error"]
        row.update({"follow_status": "error", "failed_stage": "후속 분석", "error_reason": reason})
        rows.append(row)
        continue
    if f_event["output"] is None or not str(f_event["output"]).strip():
        row.update({"follow_status": "error", "failed_stage": "후속 분석", "error_reason": "후속 결과 없음"})
        rows.append(row)
        continue

    # 5) 정상 완료
    row.update({"follow_status": "success", "status": "success", "final_result": f_event["output"]})
    rows.append(row)

result_cols = ["document_id", "text", "기존_category", "기존_summary", "기존_answer",
               "raw_category", "category", "route", "class_status", "follow_status",
               "final_result", "status", "failed_stage", "error_reason"]
workflow_results = pd.DataFrame(rows)[result_cols]
print("[workflow_results] (원문은 글자 수만, 기존 출력은 생략하고 표시)")
print(workflow_results.drop(columns=["기존_summary", "기존_answer"])
      .assign(text=workflow_results["text"].str.len().astype(str) + "자").to_string(index=False))
print("\n입력 문서 수:", len(documents), "/ 결과 행 수:", len(workflow_results),
      "/ 모든 문서가 한 행씩:", workflow_results["document_id"].tolist() == documents["document_id"].tolist())

# workflow_inputs와 follow_attempted_ids 확인
print("\n[workflow_inputs]")
for fi in workflow_inputs:
    print(fi["document_id"], "|", fi["category"], "|", fi["instruction"][:40], "... | 원문 글자 수", len(fi["text"]))
print("follow_attempted_ids:", follow_attempted_ids)
print("40005가 후속 시도에 없음:", 40005 not in follow_attempted_ids)
print("40007이 후속 시도에 없음:", 40007 not in follow_attempted_ids)

# 성공, 실패 집계
success_ids = workflow_results.loc[workflow_results["status"] == "success", "document_id"].tolist()
print("\n전체 성공 건수:", len(success_ids), "/ ID:", success_ids)
print("실패 단계별 건수:")
print(workflow_results.loc[workflow_results["status"] == "error", "failed_stage"].value_counts().to_string())

# 다시 확인할 문서 (실제 재요청은 하지 않음)
next_check = {
    40003: "분류(다수 요건 민원)는 유지하고, 요약 요청 실패 원인 확인 후 요약 단계만 재요청 검토",
    40005: "'복합 문의'가 나온 이유 확인, 분류 지시의 범주 제한을 점검한 뒤 분류부터 다시 처리",
    40007: "분류 요청 실패 원인(요청 상태, 오류 기록) 확인 후 분류부터 다시 처리",
}
review_needed = workflow_results[workflow_results["status"] == "error"].copy()
review_needed["다음 확인 사항"] = review_needed["document_id"].map(next_check)
print("\n[review_needed]")
print(review_needed[["document_id", "raw_category", "category", "route", "class_status",
                     "follow_status", "failed_stage", "error_reason", "다음 확인 사항"]].to_string(index=False))


[workflow_results] (원문은 글자 수만, 기존 출력은 생략하고 표시)
 document_id  text 기존_category raw_category category route class_status follow_status                                              final_result  status failed_stage           error_reason
       40001 2575자    다수 요건 민원     다수 요건 민원 다수 요건 민원 상담 요약      success       success 고객은 사은품 케이스, 보험 매장 코드, 카드 할인과 휴대폰 정지 횟수를 문의하고 관련 안내를 받았다. success                                    
       40003 2721자    다수 요건 민원     다수 요건 민원 다수 요건 민원 상담 요약      success         error                                                       NaN   error        후속 분석 후속 요청 실패: 요약 요청 실패(가상)
       40005 2297자    다수 요건 민원        복합 문의      NaN   NaN        error       not_run                                                       NaN   error           분류    허용되지 않은 분류 값: 복합 문의
       40006 2463자    단일 요건 민원     단일 요건 민원 단일 요건 민원  질의응답      success       success                   네. 상담사는 매장에서 해지할 때 명의자 신분증을 지참하라고 안내했다. success                                    
       40007 

### 해석 답안 작성

- Q1. 정상 완료 문서는 2개로 40001과 40006이다. 40001은 상담 요약, 40006은 질의응답 경로를 거쳐 최종 결과까지 저장되었다. 분류 단계 오류는 40005와 40007 두 건이고, 후속 분석 오류는 40003 한 건이다. 실패 단계별로 세면 분류 2건, 후속 분석 1건이고, 입력 준비와 분기 오류는 없었다.
- Q2. 두 문서 모두 분류 단계에서 오류가 나서 후속 분석을 실행할 근거가 없기 때문이다. 경로는 유효한 분류 결과가 있어야 정할 수 있는데, 두 문서는 그 값이 없거나 쓸 수 없다. 원인은 다르다. 40005는 분류 요청 자체는 completed였지만 출력이 허용 범주가 아닌 "복합 문의"라서 검증에서 멈춘 경우이고, raw_category에 그 원래 출력이 남아 있다. 40007은 분류 요청 자체가 failed라서 출력이 아예 없고(raw_category None), error_reason도 요청 실패로 기록되었다. 두 문서 모두 `follow_attempted_ids`에 들어가지 않은 것도 확인했다.
- Q3. 반드시 분류부터 다시 할 필요는 없다. 40003은 분류 요청이 completed이고 출력 "다수 요건 민원"이 허용 범주라서 class_status가 success이고, 실패한 건 요약 요청뿐이다. 그래서 document_id, 원문, raw_category와 category("다수 요건 민원"), route("상담 요약"), 그리고 `workflow_inputs`에 보관한 후속 입력을 유지해서 요약 단계만 다시 처리하면 된다. 재처리 전에는 요약 요청이 왜 실패했는지(요청 오류, 한도, 일시적 문제 등) 확인하고, 보관한 후속 입력의 원문과 분류 값이 원래 값과 같은지, 같은 문서의 결과가 중복으로 저장되지 않게 기존 행에 연결할 수 있는지를 확인해야 한다.
- Q4. 최종 결과가 있는 행만 남기면 2행만 남아서 입력 5건 중 3건이 처리되지 않았다는 사실 자체가 사라지고, 어느 단계에서 왜 실패했는지도 알 수 없게 된다. 실패 행의 category나 final_result를 CSV 기존 출력으로 채우면 모델이 만든 결과처럼 보이게 되어서, 실패했는데 성공한 것으로 집계되고 재처리 대상도 놓친다. 특히 40005는 모델이 "복합 문의"라고 답했는데 기존 출력 "다수 요건 민원"으로 채우면 분류 오류가 있었다는 기록이 없어진다. 전체 결과 연결을 검증하려면 입력 문서 수와 결과 행 수가 같은지(5 대 5), document_id가 입력 순서대로 하나씩 있는지, 단계별 상태와 실패 단계, 실패 원인이 행마다 남아 있는지, 후속 시도 ID가 분류 성공 문서와 맞는지 확인해야 한다.


---
## 실습 마무리

다음 질문에 각각 2~3문장으로 답하세요.

1. **어떤 분석 문제가 있었는가?** 작업 의존성, 조건별 경로, 중간 결과 오류와 단계별 실패를 정리하세요.
2. **어떤 방법을 적용했는가?** 단일 호출·순차 처리·조건부 분기의 선택과 입력 전달, 검증·중단·기록 방법을 설명하세요.
3. **무엇을 근거로 결론을 내렸는가?** 전달된 원문과 분류 값, 선택 경로, 단계별 상태, 최종 결과와 ID를 구분해 설명하세요.

### 제출 전 확인

- 필수 1·필수 2·과제 1의 요구사항과 Q1~Q4를 모두 작성했나요?
- 앞 단계 출력이 실제로 후속 입력에 들어가고, 원문도 함께 전달되나요?
- 미리 정한 규칙에 맞는 경로를 선택했나요?
- 허용되지 않은 분류나 분류 실패에서 후속 단계를 중단했나요?
- 실패와 미실행을 구분하고 원문·중간 결과를 보존했나요?
- 가상 응답 기반 결과를 실제 모델의 품질·비용 측정 결과로 해석하지 않았나요?


### 마무리 답안

1. 요약 단계가 분류 결과에 의존하는 작업이었고, 분류 결과에 따라 요약 또는 질의응답으로 경로가 달라져야 했다. 또 "복합 문의"처럼 허용되지 않은 중간 결과(40005), 분류 요청 실패(40007), 분류는 성공했지만 후속 요약이 실패한 경우(40003)처럼 단계마다 다른 실패가 섞여 있었다.
2. 결과끼리 의존하지 않으면 단일 호출, 앞 결과를 확인하고 다음 입력에 써야 하면 순차 처리, 결과에 따라 작업이 달라지면 조건부 분기를 골랐다. 다음 단계에는 `make_follow_input()`으로 원문과 분류 결과를 같이 넘겼고, 허용 범주가 아니거나 요청이 실패하면 `continue`로 후속 단계를 실행하지 않고 실패 단계와 원인을 기록했다.
3. 다음 단계에 전달된 원문과 분류 값이 원래 값과 일치하는지(True)를 확인했고, `choose_route()`가 정한 경로, class_status와 follow_status, failed_stage를 근거로 판단했다. 최종 결과는 document_id로 다섯 문서에 한 행씩 연결했고, success는 처리와 형식 통과라는 뜻일 뿐 원문과의 의미 일치는 사람이 따로 확인해야 한다.
